# Consultas SQL sobre silver

Consulta las tablas silver del warehouse local `data/warehouse.duckdb`, generado por `make silver`. Si no existe, correr antes `make mirror silver` (ver `CLAUDE.md`).

- Se abre en **solo lectura**: se puede tener esta notebook abierta mientras otros leen el archivo, pero `make silver` necesita que esté cerrada (`con.close()`).
- `.show()` imprime el resultado; `.df()` lo devuelve como DataFrame de pandas.
- Problemas conocidos de los datos: `docs/known_issues.md`.

In [1]:
from pathlib import Path

import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
con = duckdb.connect(str(ROOT / "data/warehouse.duckdb"), read_only=True)

In [2]:
con.sql("select product_type, count(*) from products group by 1").show()

┌──────────────────┬──────────────┐
│   product_type   │ count_star() │
│     varchar      │    int64     │
├──────────────────┼──────────────┤
│ Mortgage         │        11910 │
│ Savings Account  │       120203 │
│ Checking Account │        99979 │
│ Debit Card       │        39938 │
│ Personal Loan    │        19960 │
│ Insurance        │         2049 │
│ Credit Card      │       100102 │
│ Investment       │         5859 │
└──────────────────┴──────────────┘



## Tablas disponibles

In [3]:
con.sql("""
    select table_name, estimated_size as filas, column_count as columnas
    from duckdb_tables()
    order by estimated_size desc
""").show()

┌──────────────────────────┬─────────┬──────────┐
│        table_name        │  filas  │ columnas │
│         varchar          │  int64  │  int64   │
├──────────────────────────┼─────────┼──────────┤
│ transactions             │ 4425008 │       28 │
│ campaign_sends           │ 1746801 │       28 │
│ call_center_interactions │  686296 │       27 │
│ products                 │  400000 │       21 │
│ satisfaction_surveys     │  212759 │       26 │
│ call_transcripts         │  171321 │       24 │
│ customers                │  150000 │       31 │
│ complaints               │   67095 │       33 │
│ daily_exchange_rates     │   13164 │       11 │
│ service_agents           │    1200 │       22 │
│ branches                 │     350 │       26 │
│ marketing_campaigns      │     200 │       17 │
│ enum_mappings            │      25 │        4 │
└──────────────────────────┴─────────┴──────────┘
  13 rows                             3 columns



In [4]:
# Columnas y tipos de una tabla
con.sql("describe transactions").show(max_rows=40)

┌──────────────────────┬───────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │  column_type  │  null   │   key   │ default │  extra  │
│       varchar        │    varchar    │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼───────────────┼─────────┼─────────┼─────────┼─────────┤
│ transaction_id       │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ transaction_date     │ TIMESTAMP     │ YES     │ NULL    │ NULL    │ NULL    │
│ process_date         │ DATE          │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id           │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id          │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ transaction_type     │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ transaction_category │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ amount               │ DECIMAL(15,2) │ YES     │ NULL    │ NULL    │ NULL    │
│ currency             │ VAR

## Ejemplos

In [5]:
# Transacciones por país y estado, en USD
# amount_usd es NULL en todas las transacciones en USD (ver docs/known_issues.md): usar amount en ese caso
con.sql("""
    select transaction_country, transaction_status,
           count(*) as n,
           round(sum(coalesce(amount_usd, case when currency = 'USD' then amount end)), 2) as usd
    from transactions
    group by all
    order by n desc
    limit 10
""").show()

┌─────────────────────┬────────────────────┬─────────┬───────────────┐
│ transaction_country │ transaction_status │    n    │      usd      │
│       varchar       │      varchar       │  int64  │ decimal(38,2) │
├─────────────────────┼────────────────────┼─────────┼───────────────┤
│ Mexico              │ Approved           │ 1974254 │ 3307188088.17 │
│ Colombia            │ Approved           │ 1186505 │ 1900320196.22 │
│ Argentina           │ Approved           │  797889 │ 1276679472.23 │
│ Mexico              │ Declined           │  107596 │  179671592.00 │
│ Colombia            │ Declined           │   64177 │  102880479.34 │
│ Argentina           │ Declined           │   43492 │   69661936.90 │
│ Mexico              │ Pending            │   42766 │   72317042.27 │
│ Spain               │ Approved           │   37401 │   61654851.21 │
│ USA                 │ Approved           │   37331 │   60737672.61 │
│ Brazil              │ Approved           │   37301 │   60847088.56 │
└─────

In [6]:
# Motivos de contacto al call center, con tasa de resolución en primer contacto
# (contact_reason es la misma categoría en español, así que se agrupa solo por reason_category)
con.sql("""
    select reason_category,
           count(*) as n,
           round(100 * avg(was_resolved::int), 1) as fcr_pct
    from call_center_interactions
    group by all
    order by n desc
""").show()

┌─────────────────┬────────┬─────────┐
│ reason_category │   n    │ fcr_pct │
│     varchar     │ int64  │ double  │
├─────────────────┼────────┼─────────┤
│ Transactional   │ 240056 │    91.5 │
│ Product         │ 150863 │    89.6 │
│ Complaint       │ 117021 │    43.6 │
│ Technical       │ 102899 │    69.9 │
│ Commercial      │  54879 │    65.2 │
│ Retention       │  20578 │    60.2 │
└─────────────────┴────────┴─────────┘



In [7]:
# Últimos movimientos de un cliente, con sus productos (consulta tipo agente)
customer = con.sql("select customer_id from transactions limit 1").fetchone()[0]
con.sql(f"""
    select t.transaction_date, p.product_type, t.transaction_type, t.amount, t.currency,
           t.merchant_name, t.transaction_status
    from transactions t
    join products p using (product_id)
    where t.customer_id = '{customer}'
    order by t.transaction_date desc
    limit 10
""").show()

┌─────────────────────┬───────────────┬──────────────────┬───────────────┬──────────┬────────────────────┬────────────────────┐
│  transaction_date   │ product_type  │ transaction_type │    amount     │ currency │   merchant_name    │ transaction_status │
│      timestamp      │    varchar    │     varchar      │ decimal(15,2) │ varchar  │      varchar       │      varchar       │
├─────────────────────┼───────────────┼──────────────────┼───────────────┼──────────┼────────────────────┼────────────────────┤
│ 2026-05-07 18:05:30 │ Personal Loan │ Adjustment       │        220.48 │ USD      │ NULL               │ Approved           │
│ 2026-04-07 02:35:33 │ Debit Card    │ Purchase         │         14.55 │ USD      │ Tienda General     │ Approved           │
│ 2026-01-29 21:58:46 │ Personal Loan │ Payment          │        699.76 │ USD      │ NULL               │ Approved           │
│ 2025-12-20 10:06:50 │ Debit Card    │ Purchase         │        408.86 │ USD      │ Cable TV          

In [8]:
# Como DataFrame de pandas
df = con.sql("select segment, country, count(*) as clientes from customers group by all order by 1, 2").df()
df

,segment,country,clientes
0,Basic,Argentina,17924
1,Basic,Colombia,27069
2,Basic,Mexico,44763
3,Plus,Argentina,7441
4,Plus,Colombia,11309
5,Plus,Mexico,18797
6,Premium,Argentina,2979
7,Premium,Colombia,4584
8,Premium,Mexico,7644
9,Student,Argentina,1498


## Calidad de datos por fila

`_dq_issues` lista los problemas de cada fila frente al diccionario (`cast_failed`, `not_null`, `undocumented_value`, `out_of_range`).

In [9]:
con.sql("""
    select issue, count(*) as filas
    from (select unnest(_dq_issues) as issue from call_center_interactions)
    group by 1
    order by 2 desc
""").show()

┌───────────────────────────────────────┬───────┐
│                 issue                 │ filas │
│                varchar                │ int64 │
├───────────────────────────────────────┼───────┤
│ undocumented_value:reason_category    │ 20578 │
│ undocumented_value:detected_sentiment │ 18973 │
│ undocumented_value:channel            │  3395 │
└───────────────────────────────────────┴───────┘



## Leer el Parquet del lake en vez del warehouse

Mismo contenido, en `data/lake/silver/` (o en S3, publicado con `make publish`). Útil si no tienes el `.duckdb`.

In [10]:
con.sql(f"""
    select year, count(*) as filas
    from read_parquet('{ROOT}/data/lake/silver/transactions/*/*/*.parquet', hive_partitioning = true)
    group by 1
    order by 1
""").show()

┌───────┬─────────┐
│ year  │  filas  │
│ int64 │  int64  │
├───────┼─────────┤
│  2023 │  801170 │
│  2024 │ 1471814 │
│  2025 │ 1466502 │
│  2026 │  685522 │
└───────┴─────────┘

